In [25]:
import os
import squidpy as sq
import scanpy as sc
import anndata as ad
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [26]:
in_dir = "../../data/10x_visium"
out_dir = "../../results/10_compare_with_public_data/01_create_pseudo_bulk"
os.makedirs(out_dir, exist_ok=True)

In [27]:
sample_list = ['15837', '15852', '16512', '21022', '8262', 
               'B23-03794-5', 'B23-05785-7-1', 'B23-05785-7-2', 
               'X1842', 'X1845', 'X1899', 'X1912']

In [28]:
df_list = []
for sample in sample_list:
    adata = sq.read.visium(f"{in_dir}/{sample}")

    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    adata.obs['sample'] = f"sample_{sample}"

    # Sum counts across all spots for each sample
    pseudo_bulk = adata.to_df().groupby(adata.obs['sample']).sum()
    pseudo_bulk = pseudo_bulk.T  # Transpose to have genes as rows and samples as columns
    pseudo_bulk = pseudo_bulk.astype('int')  # Convert to int to save memory
    df_list.append(pseudo_bulk)

In [29]:
df = pd.concat(df_list, axis=1)

In [30]:
df.head()

sample,sample_15837,sample_15852,sample_16512,sample_21022,sample_8262,sample_B23-03794-5,sample_B23-05785-7-1,sample_B23-05785-7-2,sample_X1842,sample_X1845,sample_X1899,sample_X1912
SAMD11,113,260,172,573,24,583,862,1108,764,333,1041,911
NOC2L,191,257,374,618,52,687,1228,1260,1808,628,1570,2588
KLHL17,80,129,134,201,33,353,679,460,777,252,636,1109
PLEKHN1,53,88,60,144,34,93,333,217,348,93,285,634
PERM1,12,45,28,27,11,125,302,193,122,25,105,100


In [31]:
df.to_csv(f"{out_dir}/pseudo_bulk_counts.csv")